# NVTTS Evaluator: đánh giá một model TTS (chỉ chạy đánh giá)

Notebook này **chỉ chạy đánh giá**. Phần hiệu chuẩn (huấn luyện detector NVPA, chọn dung sai và ngưỡng) đã làm xong và nằm trong
dataset checkpoint `nvtts-eval-ckpt`; cấu hình đánh giá được lấy thẳng từ checkpoint đó nên kết quả của model luôn so sánh được với
"trần" của audio thật.

**Chuẩn bị (panel bên phải):**
1. **Add Input** → `vinv-tts-train` (dữ liệu `vinv-tts/train` và `vinv-tts/dev`).
2. **Add Input** → `nvtts-eval-ckpt` (gồm `nvpa_gap.joblib` và `summary.json` của audio thật).
3. **Add Input** → dataset chứa **output của model TTS cho 316 câu dev**: mỗi mẫu một file `<spk_id>_<tên file audio>.wav`,
   ví dụ `spk_0000_0001.wav` (xuất danh sách văn bản đầu vào bằng ô ở bước 6 nếu cần).
4. **Settings → Internet: On** (cần cho `git`, `pip` và tải model). GPU T4 là tùy chọn (nhanh hơn DNSMOS/ECAPA; ASR luôn chạy trên CPU).

Thời gian: khoảng 8 đến 10 phút cho 316 mẫu. Mọi ô đều chạy lại an toàn; kết quả đã tính được lưu cache.

### 1. Tham số (chỉ cần sửa ô này)

In [ ]:
# ================= SỬA 2 DÒNG NÀY =================
GEN = "/kaggle/input/datasets/<tai-khoan>/<dataset-output-model>"   # thư mục chứa spk_0000_0001.wav, ...
RUN_NAME = "mymodel_A"                                              # tên lần chạy; kết quả nằm ở /kaggle/working/runs/<RUN_NAME>
# ===================================================

DATA = "/kaggle/input/datasets/hongvitchugo/vinv-tts-train/vinv-tts"
CKPT = "/kaggle/input/datasets/hongvitchugo/nvtts-eval-ckpt"
REPO = "/kaggle/working/NVTTS-Evaluator"
ASR_DIR = "/kaggle/working/models/Zipformer-30M-RNNT-6000h"
MANIFEST = f"/kaggle/working/manifests/{RUN_NAME}.jsonl"
RUN_DIR = f"/kaggle/working/runs/{RUN_NAME}"
CFG = "/kaggle/working/kaggle_eval.yaml"
ALLOW_MISSING = False        # False: dừng nếu model thiếu mẫu nào (mẫu thiếu bị loại khỏi chỉ số, làm điểm không còn so sánh được)

import subprocess
from pathlib import Path

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

### 2. Kiểm tra đầu vào (dừng sớm nếu thiếu)

In [ ]:
problems = []
if "<" in GEN or not Path(GEN).is_dir():
    problems.append(f"GEN chưa đúng hoặc không tồn tại: {GEN}  (sửa ở bước 1; xem tên dataset ở /kaggle/input)")
for name, p in (("DATA", DATA), ("CKPT", CKPT)):
    if not Path(p).is_dir():
        problems.append(f"{name} không tồn tại: {p}  (đã Add Input chưa?)")
detectors = sorted(Path(CKPT).rglob("nvpa_gap.joblib")) if Path(CKPT).is_dir() else []
refs = sorted(Path(CKPT).rglob("summary.json")) if Path(CKPT).is_dir() else []
if Path(CKPT).is_dir() and not detectors:
    problems.append("Không thấy nvpa_gap.joblib trong CKPT")
if Path(CKPT).is_dir() and not refs:
    problems.append("Không thấy summary.json (kết quả audio thật) trong CKPT")
if problems:
    raise ValueError("\n".join(problems))

DETECTOR, REF_SUMMARY = str(detectors[0]), str(refs[0])
audio = [p for p in Path(GEN).iterdir() if p.suffix.lower() in (".wav", ".flac")]
print("detector      :", DETECTOR)
print("kết quả thật  :", REF_SUMMARY)
print(f"audio trong GEN: {len(audio)} file (cần 316 cho tập dev); ví dụ: {sorted(p.name for p in audio)[:3]}")
if not audio:
    print("GEN không có file audio trực tiếp. Tìm thử một file mẫu:")
    for p in list(Path("/kaggle/input").rglob("spk_0000_0001.*"))[:5]:
        print("  tìm thấy:", p, "  -> đặt GEN =", p.parent)

### 3. Cài đặt

In [ ]:
!test -d {REPO}/.git && git -C {REPO} pull || git clone --depth 1 https://github.com/VietCH57/NVTTS-Evaluator.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q -e ".[audio,config,asr,mos,asv,nvpa]"
import sys
sys.path.insert(0, REPO)

### 4. Mô hình ASR (tải lần đầu mỗi session)

In [ ]:
from huggingface_hub import snapshot_download

if not (Path(ASR_DIR) / "tokens.txt").exists():
    snapshot_download("hynt/Zipformer-30M-RNNT-6000h",
        allow_patterns=["encoder-epoch-20-avg-10.onnx", "decoder-epoch-20-avg-10.onnx",
                        "joiner-epoch-20-avg-10.onnx", "bpe.model"],
        local_dir=ASR_DIR)
    sh(f"python scripts/make_tokens.py --bpe {ASR_DIR}/bpe.model --out {ASR_DIR}/tokens.txt")
print(sorted(p.name for p in Path(ASR_DIR).glob("*")))     # phải có tokens.txt

### 5. Cấu hình lấy từ checkpoint
Cấu hình đánh giá (chuẩn hóa văn bản, dung sai NVPA, hệ số ngưỡng, cách tính pMOS/SS...) được **sao nguyên từ `summary.json` của audio thật**,
chỉ đổi các đường dẫn. Nhờ vậy kết quả model và "trần" được tính hoàn toàn cùng một cách. Ô này cũng kiểm tra detector khớp với summary tham chiếu.

In [ ]:
import json
import yaml
from nvtts_eval.config import config_from_dict
from nvtts_eval.nvpa.detector import GapDetector

ref = json.load(open(REF_SUMMARY, encoding="utf-8"))
cfg = ref["active_assumptions"]
cfg["asr"]["model_dir"] = ASR_DIR
cfg["ss"]["savedir"] = "/kaggle/working/models/spkrec-ecapa-voxceleb"
cfg["nvpa"]["detector_path"] = DETECTOR
config_from_dict(cfg)                                        # kiểm tra hợp lệ
Path(CFG).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")

det = GapDetector.load(Path(DETECTOR))                       # cảnh báo nếu khác phiên bản scikit-learn lúc huấn luyện
ref_thr = ref["automatic_metrics"]["nvpa"]["detector"]["thresholds"]
mismatch = {t: (det.thresholds[t], v) for t, v in ref_thr.items() if abs(det.thresholds[t] - v) > 1e-5}
assert not mismatch, f"detector không khớp summary tham chiếu (khác file checkpoint?): {mismatch}"
n = cfg["nvpa"]
print(f"NVPA: dung sai trước/sau = {n['tolerance_before']}/{n['tolerance_after']} từ, hệ số ngưỡng x{n['threshold_scale']}  |  detector khớp summary tham chiếu")

### 6. Manifest từ output của model
Ghép từng câu dev với file `<sample_id>.wav` trong `GEN`. Ô dưới cũng xuất `dev_inputs.jsonl` (văn bản có tag, `sample_id`, `speaker_id`)
để bạn đưa vào model TTS nếu chưa sinh audio.

In [ ]:
sh(f"python -m nvtts_eval.data.adapter_vinv --root {DATA} --split dev --track A --source model --generated-dir {GEN} --out {MANIFEST}")

from nvtts_eval.data import Manifest
man = Manifest.load(Path(MANIFEST))
missing = [s.sample_id for s in man if s.generated_audio is None]
extra = sorted({p.stem for p in audio} - {s.sample_id for s in man})
print(f"\nmẫu cần đánh giá: {len(man)} | thiếu audio: {len(missing)} | file thừa (tên không khớp mẫu nào): {len(extra)}")
if missing:
    print("  ví dụ mẫu thiếu:", missing[:5])
if extra:
    print("  ví dụ file thừa :", extra[:5])
if missing and not ALLOW_MISSING:
    raise ValueError(f"Model thiếu {len(missing)}/{len(man)} mẫu. Mẫu thiếu bị loại khỏi chỉ số nên điểm không so sánh được. "
                     f"Sinh đủ audio, hoặc đặt ALLOW_MISSING = True nếu cố ý.")

# văn bản đầu vào cho model TTS (có thẻ NV), mỗi dòng một câu
with open("/kaggle/working/dev_inputs.jsonl", "w", encoding="utf-8") as f:
    for s in man:
        f.write(json.dumps({"sample_id": s.sample_id, "speaker_id": s.speaker_id, "text": s.text}, ensure_ascii=False) + "\n")

### 7. Chạy đánh giá (ASR, pMOS, SS, NVPA)

In [ ]:
!python -m nvtts_eval.cli run --manifest {MANIFEST} --run-dir {RUN_DIR} --config {CFG}

### 8. Kết quả so với audio thật
"Audio thật" là dev ground truth đóng vai output model (cùng cấu hình), tức mức tối đa thực tế. Điểm tự động **không phải điểm chính thức**
(thiếu SN/Q của người chấm); NVPA thì đọc cùng cột NV thừa/100 từ và mức ngẫu nhiên (model chèn NV tràn lan sẽ được NVPA cao).

In [ ]:
s = json.load(open(f"{RUN_DIR}/summary.json", encoding="utf-8"))

def pick(d):
    m = d["automatic_metrics"]; nv = m["nvpa"]
    return {"NVPA": nv["value"], "WER": m["wer"]["value"], "pMOS": m["pmos"]["mean"], "SS": m["ss"]["mean"],
            "random": nv["shuffle_baseline"]["mean"], "spurious": nv["spurious_per_100_words"],
            "auto": d["automatic_score"]["value"] if d["automatic_score"] else None}

a, r = pick(s), pick(ref)
rows = [("NVPA", "cao hơn tốt"), ("WER", "thấp hơn tốt"), ("pMOS", "cao hơn tốt"), ("SS", "cao hơn tốt")]
print(f"{'chỉ số':8s} {'model':>8s} {'audio thật':>11s}   ghi chú")
for k, note in rows:
    print(f"{k:8s} {a[k]:8.3f} {r[k]:11.3f}   {note}")
print(f"{'AutoScore':8s} {a['auto']:8.3f} {r['auto']:11.3f}   tối đa 0.70, không chính thức" if a["auto"] is not None else "AutoScore: N/A")
print(f"\nNVPA: mức ngẫu nhiên {a['random']:.3f} (thật {r['random']:.3f}); lift = {a['NVPA'] - a['random']:.3f} (thật {r['NVPA'] - r['random']:.3f})")
print(f"NV thừa/100 từ: {a['spurious']:.2f} (thật {r['spurious']:.2f}; mật độ NV yêu cầu ~5.4)")

nv, nvr = s["automatic_metrics"]["nvpa"], ref["automatic_metrics"]["nvpa"]
print("\nNVPA theo loại (model / thật, n):")
for t, v in nv["by_type"].items():
    flag = "  <- n nhỏ, đọc thận trọng" if v["low_n"] else ""
    print(f"  {t:16s} {v['mean']:.3f} / {nvr['by_type'][t]['mean']:.3f}   n={v['n']}{flag}")
print("lý do trượt:", {k: v for k, v in nv["reasons"].items() if k != "ok"})
print("kiểm tra dung sai (NVPA/ngẫu nhiên):", {c["tolerance"]: (round(c["nvpa"], 3), round(c["random"], 3)) for c in nv["tolerance_check"]})
w, p, q = s["automatic_metrics"]["wer"], s["automatic_metrics"]["pmos"], s["automatic_metrics"]["ss"]
print(f"\nKhoảng tin cậy 95% (theo câu / theo speaker): NVPA {nv['ci_utterance'][0]:.3f}-{nv['ci_utterance'][1]:.3f} / {nv['ci_speaker'][0]:.3f}-{nv['ci_speaker'][1]:.3f}")
print(f"  mẫu lỗi: ASR {w['n_failed']}, pMOS {p['n_failed']}, SS {q['n_failed']}, NVPA {nv['n_failed']}", "| WER có chữ số trong output ASR:", w["n_hyp_with_digits"])

### 9. Chi tiết từng mẫu (để tìm lỗi của model)

In [ ]:
import pandas as pd
from nvtts_eval.config import load_config
from nvtts_eval.core.artifacts import ArtifactStore
from nvtts_eval.metrics.wer import wer_records

store, cfg_obj = ArtifactStore(RUN_DIR), load_config(Path(CFG))
w = pd.DataFrame(wer_records(man, store.read_records("asr"), cfg_obj.text_norm))
w = w.reindex(columns=["sample_id", "speaker_id", "ref_words", "errors", "wer", "hyp"])
pm = pd.DataFrame(store.read_records("pmos")).reindex(columns=["sample_id", "ovrl"])
ss = pd.DataFrame(store.read_records("ss")).reindex(columns=["sample_id", "cosine"])
nvdf = pd.DataFrame([{"sample_id": x["sample_id"], "nv_events": len(x["events"]), "nv_hits": sum(e["hit"] for e in x["events"]),
                      "nv_spurious": len(x["spurious"])} for x in store.read_records("nvpa") if "error" not in x])
df = w.merge(pm, on="sample_id", how="left").merge(ss, on="sample_id", how="left").merge(nvdf, on="sample_id", how="left")
df["nvpa_rate"] = df["nv_hits"] / df["nv_events"]
Path("/kaggle/working/results").mkdir(exist_ok=True)
df.to_csv(f"/kaggle/working/results/{RUN_NAME}_per_sample.csv", index=False, encoding="utf-8-sig")
print("10 câu WER cao nhất:");              display(df.sort_values("wer", ascending=False).head(10)[["sample_id", "wer", "ref_words", "hyp"]])
print("10 câu NVPA thấp nhất (>=3 sự kiện):"); display(df[df["nv_events"] >= 3].sort_values("nvpa_rate").head(10)[["sample_id", "nv_events", "nv_hits", "nv_spurious"]])
print("Theo speaker (>=5 câu): SS và NVPA trung bình")
g = df.groupby("speaker_id").agg(n=("sample_id", "count"), SS=("cosine", "mean"), NVPA=("nvpa_rate", "mean"), WER=("wer", "mean"))
display(g[g["n"] >= 5].sort_values("n", ascending=False).round(3))

### 10. Lưu kết quả

In [ ]:
import shutil
out = Path("/kaggle/working/results"); out.mkdir(exist_ok=True)
shutil.copy(f"{RUN_DIR}/summary.json", out / f"{RUN_NAME}_summary.json")
shutil.make_archive(f"/kaggle/working/{RUN_NAME}_results", "zip", "/kaggle/working/runs", RUN_NAME)   # toàn bộ artifact từng mẫu
print(sorted(p.name for p in out.iterdir()), "+", f"{RUN_NAME}_results.zip")

---
## (Tùy chọn) Đánh giá bởi người: thêm SN, Q và điểm cuối
Chỉ gửi cho người chấm thư mục `for_raters`; **giữ kín `PRIVATE_key.json`** (tải về ngay sau khi xuất, vì cần khi nhập điểm).
Mỗi người chấm điền một bản `rating_sheet.csv` và gửi lại đặt tên theo người chấm (`alice.csv`, `bob.csv`).
Nếu họ điền cả cột `NV_placement`, báo cáo in thêm tương quan Spearman giữa NVPA và điểm của người: phép kiểm chứng detector đáng tin nhất.

In [ ]:
RUN_HUMAN_EXPORT = False     # đổi thành True để tạo gói chấm điểm
PKG = f"/kaggle/working/human_package_{RUN_NAME}"
if RUN_HUMAN_EXPORT:
    sh(f"python -m nvtts_eval.cli human-subset --manifest {MANIFEST} --run-dir {RUN_DIR} --config {CFG}")
    sh(f"python -m nvtts_eval.cli human-export --manifest {MANIFEST} --run-dir {RUN_DIR} --config {CFG} --out {PKG}")
    shutil.make_archive(f"/kaggle/working/for_raters_{RUN_NAME}", "zip", PKG, "for_raters")
    print(f"Tải về: /kaggle/working/for_raters_{RUN_NAME}.zip (gửi người chấm) và {PKG}/PRIVATE_key.json (giữ kín)")

In [ ]:
RUN_HUMAN_IMPORT = False     # đổi thành True sau khi có điểm
RATINGS = "/kaggle/working/ratings"      # chứa alice.csv, bob.csv ... (upload vào đây hoặc dùng một Kaggle Dataset)
if RUN_HUMAN_IMPORT:
    sheets = " ".join(str(p) for p in sorted(Path(RATINGS).glob("*.csv")))
    assert sheets, f"không thấy file .csv nào trong {RATINGS}"
    sh(f"python -m nvtts_eval.cli human-import --manifest {MANIFEST} --run-dir {RUN_DIR} --config {CFG} --package {PKG} --ratings {sheets}")